# 二分 K-Means 学习笔记

状态：已在引导支持下完成一次。

## 1. 问题抽象

维护所有点的簇划分。每一步都对所有非单节点簇评估一次 2-means 拆分，并提交使 SSE 降幅最大的拆分。

## 2. 我的初始理解

外层循环决定拆分哪个簇；内层 2-means 循环交替执行 Assignment 与质心 Update。两者是相互独立的状态过程，临时最佳状态不能泄漏到下一轮外层迭代。

## 3. 算法拆解

1. 计算距离平方、质心与 SSE。
2. 使用确定性的 2-means 拆分一个簇。
3. 使用 `parent_sse - child_1_sse - child_2_sse` 评估每个候选。
4. 应用最佳拆分，并按降序输出所有簇的大小。
5. 重复上述步骤，直到达到目标簇数量。

## 4. 关键公式 / Tensor Shape

对于点 $p=(x,y)$ 与质心 $c=(c_x,c_y)$：$d^2(p,c)=(x-c_x)^2+(y-c_y)^2$。

对于簇 $C$：$SSE(C)=\sum_{p \in C} d^2(p, centroid(C))$。

拆分收益：$SSE(parent)-SSE(child_1)-SSE(child_2)$。

## 5. 伪代码

```text
clusters = [all_points]
while len(clusters) < target:
    重置最佳候选
    遍历每个可拆分簇:
        child_1, child_2 = two_means(cluster)
        gain = 父簇 SSE - 两个子簇 SSE
        按 gain 与 tie 规则更新最佳候选
    用两个子簇替换最佳父簇
    输出排序后的簇大小
```

## 6. 分步实现

最终可提交实现保存在 `solution.py` 中。复习 Assignment、Update、收敛或拆分选择时，在这里重新运行小型手算示例。

## 7. 已发现的错误

- 混淆了标量坐标与完整 point。
- 没有在每轮外层迭代开始时重置临时最佳拆分状态。
- 处理收敛与 gain 并列时需要使用浮点容差。
- 需要跳过单节点簇。
- 输入解析中必须移除交互提示和题目未要求的输出。

## 8. 学到的 Python 模式

- 只需要元素值时使用 `for point in points`。
- 同时需要索引与元素值时使用 `enumerate(clusters)`。
- 使用 `max(points, key=lambda point: point[0])` 按指定坐标选择 point。
- 使用 `map(float, input().split())` 解析符合 Online Judge 要求的坐标输入。

## 9. 最终理解

Bisecting K-Means 在标准 2-means 内层优化之外增加了贪心的外层选择。正确实现的关键，是将候选拆分评估与实际提交拆分保持分离，并应用确定性的 tie 规则。

## 10. 复习记录

下一步：在 2026-09-19 进行空白重写。与 `solution.py` 对比前，先检查状态重置、单节点跳过、容差、输入输出格式与 equal-gain 行为。